# Предобработка и анализ данных о продажах автомобилей в Казахстане (2019)

**Задача:** очистить сырые данные о продажах за 9 месяцев 2019 года и оценить положение автоцентра «Меркур Авто» на рынке.

**Артефакты:** очищенный датасет `autokz2019_clean.csv`, данный ноутбук и отчёт.

## 1. Загрузка и первичный осмотр данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# Важно: разделитель ';' и десятичный разделитель ',' — иначе числа не распознаются
df = pd.read_csv('autokz2019.csv', sep=';', decimal=',')

# Убираем случайные пробелы по краям строковых значений
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].astype(str).str.strip()

print('Размер:', df.shape)
df.head()

In [ ]:
# Типы данных и пропуски
df.info()
print()
print('Пропуски по столбцам:')
print(df.isna().sum().sort_values(ascending=False).head(12))

## 2. Глубокая очистка и преобразование данных

In [ ]:
# 2.1 Удаляем столбцы без аналитической ценности / с критичным числом пропусков
cols_to_drop = ['Модификация', 'Сегмент', 'Сегментация Eng', 'Локализация производства',
                'Тип клиента', 'Форма расчета', 'Наименование дилерского центра']
df = df.drop(columns=cols_to_drop)

# Переименовываем столбцы по-английски
df = df.rename(columns={
    'Год':'year', 'Месяц':'month', 'Компания':'dealer_company', 'Бренд':'brand',
    'Модель':'model', 'Год выпуска':'prod_year', 'Страна-производитель':'prod_country',
    'Вид топлива':'fuel_type', 'Объём двиг, л,':'engine_vol', 'Коробка передач':'gearbox',
    'Тип привода':'drive', 'Регион':'region', 'Количество':'quantity',
    'Цена, USD':'price_usd', 'Продажа, USD':'total_usd', 'Область':'oblast',
    'Сегментация 2013':'segment_2013', 'Класс 2013':'class_2013'})
print('Столбцов стало:', df.shape[1])
df.columns.tolist()

In [ ]:
# 2.2 Полностью пустые строки, возвраты (quantity = -1) и явные дубликаты
df = df.dropna(how='all')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
n_returns = int((df['quantity'] < 0).sum())
df = df[df['quantity'] > 0]          # возвраты удаляем — это не продажи
df = df.drop_duplicates().reset_index(drop=True)
print('Возвратов удалено:', n_returns, '| Строк после чистки:', df.shape)

In [ ]:
# 2.3 Числовые столбцы: запятая -> точка ДО конверсии, иначе дробные цены теряются (NaN)
for c in ['price_usd', 'total_usd']:
    df[c] = (df[c].astype(str)
             .str.replace('\u00a0', '', regex=False)
             .str.replace(' ', '', regex=False)
             .str.replace(',', '.', regex=False))
    df[c] = pd.to_numeric(df[c], errors='coerce')

print('NaN в ценах:', df['price_usd'].isna().sum())
print('Сходимость total_usd = price * quantity:',
      bool(np.allclose(df['total_usd'], df['price_usd'] * df['quantity'])))

In [ ]:
# 2.4 Восстановление смещённых строк: у части Renault в 'fuel_type' стоит объём,
# а в 'engine_vol' — коробка передач. Определяем их и сдвигаем значения на место.
def is_num(s):
    try:
        float(str(s).replace(',', '.'))
        return True
    except ValueError:
        return False

shift_mask = (df['brand'].isin(['Renault'])) & (df['fuel_type'].apply(is_num)) & (~df['engine_vol'].apply(is_num))
print('Смещённых строк:', shift_mask.sum())
df.loc[shift_mask, 'engine_vol'] = df.loc[shift_mask, 'fuel_type']
df.loc[shift_mask, 'gearbox']    = df.loc[shift_mask, 'drive']
df.loc[shift_mask, 'drive']      = np.nan      # исходное значение привода утрачено
df.loc[shift_mask, 'fuel_type']  = np.nan

In [ ]:
# 2.5 Страна производства -> коды ALPHA-3
country_map = {
    'Германия':'DEU', 'Россия':'RUS', 'Казахстан':'KAZ', 'США':'USA', 'Япония':'JPN',
    'Корея':'KOR', 'Южная Корея':'KOR', 'Китай':'CHN', 'Франция':'FRA', 'Чехия':'CZE',
    'Словакия':'SVK', 'Венгрия':'HUN', 'Турция':'TUR', 'Испания':'ESP', 'Италия':'ITA',
    'Великобритания':'GBR', 'Румыния':'ROU', 'Узбекистан':'UZB', 'Швеция':'SWE',
    'Австрия':'AUT', 'Бельгия':'BEL', 'Польша':'POL'}
df['prod_country'] = df['prod_country'].map(country_map).fillna(df['prod_country'])
df['prod_country'].value_counts().head(8)

In [ ]:
# 2.6 Вид топлива -> F (бензин), D (дизель), E (электро), HYB (гибрид)
fuel_map = {
    'бензин':'F', 'Бензин':'F', 'БЕНЗИН':'F', 'дизель':'D', 'Дизель':'D', 'ДИЗЕЛЬ':'D',
    'электро':'E', 'Электро':'E', 'Электричество':'E', 'гибрид':'HYB', 'Гибрид':'HYB',
    'гибридный':'HYB', 'plug-in гибрид':'HYB', 'Plug-in Hybrid':'HYB',
    'гибрид бензин':'HYB', 'гибрид дизель':'HYB', 'газобаллонное':'F', 'LPG':'F'}
df['fuel_type'] = df['fuel_type'].map(lambda x: fuel_map.get(x, x if x in ['F','D','E','HYB'] else np.nan))
df['fuel_type'].value_counts(dropna=False)

In [ ]:
# 2.7 Тип привода: ~26 вариантов написания -> 4WD / FWD / RWD
drive_map = {
    '4WD':'4WD', 'AWD':'4WD', 'Полный':'4WD', 'полный':'4WD', 'ПОЛНЫЙ':'4WD',
    'Полный привод':'4WD', '4х4':'4WD', '4x4':'4WD', '4х4.2':'4WD',
    '4WD (подключаемый)':'4WD', 'Part-time':'4WD',
    'FWD':'FWD', 'передний':'FWD', 'Передний':'FWD', 'ПЕРЕДНИЙ':'FWD',
    'Передний привод':'FWD', '2WD':'FWD', '4х2':'FWD', '4x2':'FWD', '2.2':'FWD',
    'Переднеприводной':'FWD',
    'RWD':'RWD', 'задний':'RWD', 'Задний':'RWD', 'ЗАДНИЙ':'RWD',
    'Задний привод':'RWD', 'Заднеприводной':'RWD'}
df['drive'] = df['drive'].map(lambda x: drive_map.get(x, x if x in ['4WD','FWD','RWD'] else np.nan))
df['drive'].value_counts(dropna=False)

In [ ]:
# 2.8 Коробка передач: 83 варианта (в т.ч. смешанные алфавиты: '6АТ', 'АT', 'CVT(вAриATор)')
# -> две категории. Мусор, попавший не в тот столбец ('4WD', 'TDI'), -> NaN.
def norm_gb(s):
    if not isinstance(s, str):
        return np.nan
    t = s.strip()
    if t == '':
        return np.nan
    u = t.upper()
    if u in ('4WD', 'FWD', 'RWD', '2WD', 'TDI', 'ТДИ'):
        return np.nan
    if re.search(r'АМТ|AMT', u):                      # робот
        return 'Автомат'
    if re.search(r'М[ЕЕ]Х|МКП', u):                   # Мех., МКПП
        return 'Механика'
    # МТ / MT — кириллица или латиница, с цифрами и без
    if (re.match(r'^\d*\s*[МM]\s*[/\-]?\s*[ТT]$', u)
            or re.match(r'^\d*\s*[МM]\s*$', u)
            or re.match(r'^\d*[МM][ТT]$', u)):
        return 'Механика'
    # AT / АТ / 6 AT / 4AT / A/T и т.п.
    if (re.match(r'^\d*\s*[АA]?[ТT]$', u)
            or re.match(r'^\d*\s*[АA]\s*/?\s*[ТT]\s*$', u)
            or re.match(r'^\d*\s*[АA]$', u)):
        return 'Автомат'
    if re.search(r'АКП|АКПП|DSG|DCT|CVT|TRONIC|PDK|ВАРИАТОР|РОБОТ|АВТОМАТ|AUTOMATIC', u):
        return 'Автомат'
    return np.nan

df['gearbox'] = df['gearbox'].map(norm_gb)
df['gearbox'].value_counts(dropna=False)

In [ ]:
# 2.9 Объём двигателя: запятая -> точка, литеры 'L', куб. см -> литры,
# мощность вместо объёма (Jaguar I-Pace '400 Л.С.', электромобиль) -> NaN,
# арифметические аномалии Chevrolet Niva (11,7...26,7) -> реальный объём 1.7 л
def clean_vol(v, brand, model):
    s = str(v).strip()
    if s in ('nan', ''):
        return np.nan
    s = s.replace(',', '.').replace('л', '').replace('L', '').strip()
    if re.search(r'л\.?с|hp|kwh', s, re.I):      # мощность или кВтч — это не объём
        return np.nan
    try:
        x = float(s)
    except ValueError:
        return np.nan
    if x >= 1000:                                  # куб. см -> литры
        x = x / 1000
    # большие значения легитимны только у грузовиков
    if x > 12 and not (brand == 'Scania' or (brand == 'Mercedes-Benz' and model == 'Actros')):
        return np.nan
    if brand == 'Chevrolet' and model == 'Niva' and x != 1.7:
        x = 1.7                                    # исправляем аномалии 11,7 / 21,6 / 26,7
    return x

df['engine_vol'] = [clean_vol(v, b, m)
                    for v, b, m in zip(df['engine_vol'], df['brand'], df['model'])]
df['engine_vol'].describe()

In [ ]:
# 2.10 Дата продажи: последний день месяца из столбцов year и month
MONTHS = {'Январь':1, 'Февраль':2, 'Март':3, 'Апрель':4, 'Май':5, 'Июнь':6,
          'Июль':7, 'Август':8, 'Сентябрь':9, 'Октябрь':10, 'Ноябрь':11, 'Декабрь':12}
df['month'] = df['month'].map(MONTHS)
df['sale_date'] = pd.to_datetime(dict(year=df['year'], month=df['month'], day=1)) + pd.offsets.MonthEnd(0)
df = df.drop(columns=['year', 'month'])
df['prod_year'] = pd.to_numeric(df['prod_year'], errors='coerce')
df['sale_date'].dt.to_period('M').value_counts().sort_index()

In [ ]:
# 2.11 Неявные дубликаты названий компаний
comp_map = {
    'Mercur Autos':'Mercur Auto', 'MERCUR AUTO':'Mercur Auto',
    'Ravon Motors Kazakstan':'Равон Моторс Казахстан', 'Автокапитал':'Autokapital',
    'Каспиан Моторс':'Caspian Motors', 'ММС Рус':'MMC RUS'}
df['dealer_company'] = (df['dealer_company'].replace(comp_map)
                        .str.replace('  ', ' ', regex=False).str.strip())
df['dealer_company'].nunique()

In [ ]:
# 2.12 Регион и область: нормальный регистр, срезаем префикс 'г.'
def cap_ru(s):
    if not isinstance(s, str):
        return s
    s = re.sub(r'^г\.\s*', '', s)
    return '-'.join((w[0].upper() + w[1:]) if w else w for w in s.split(' '))

df['region'] = df['region'].map(cap_ru)
df['oblast'] = df['oblast'].map(cap_ru)
df['region'].value_counts().head(8)

In [ ]:
# 2.13 Финальные типы: категориальные признаки + сохранение датасета
# (кодировка utf-8-sig — чтобы Excel открывал кириллицу без кракозябр)
for c in ['fuel_type', 'gearbox', 'drive', 'segment_2013', 'class_2013']:
    df[c] = df[c].astype('category')

df.to_csv('autokz2019_clean.csv', index=False, encoding='utf-8-sig', sep=';')
print('Итоговый размер:', df.shape)
df.info()

## 3. Исследовательский анализ данных (EDA)

In [ ]:
# 3.1 Распределение цены
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['price_usd'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Цена (весь диапазон)')
axes[0].set_xlabel('USD')
axes[1].hist(df.loc[df['price_usd'] <= 60000, 'price_usd'], bins=60, color='steelblue')
axes[1].set_title('Цена до 60 тыс. USD (95% рынка)')
axes[1].set_xlabel('USD')
plt.tight_layout()
print(f"Средняя цена: {df['price_usd'].mean():,.0f} USD | Медиана: {df['price_usd'].median():,.0f} USD")

In [ ]:
# 3.2 Год выпуска и объём двигателя
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['prod_year'].dropna(), bins=25, color='darkseagreen', edgecolor='white')
axes[0].set_title('Год выпуска')
axes[1].hist(df['engine_vol'].dropna(), bins=40, color='tan', edgecolor='white')
axes[1].set_title('Объём двигателя, л')
plt.tight_layout()
print('Медианный год выпуска:', df['prod_year'].median())
print('Медианный объём двигателя:', df['engine_vol'].median())

In [ ]:
# 3.3 Топ-10 брендов и топ-10 моделей рынка
top_br = df.groupby('brand')['quantity'].sum().sort_values(ascending=False).head(10)
top_md = df.groupby(['brand', 'model'])['quantity'].sum().sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(top_br.index[::-1], top_br.values[::-1], color='steelblue')
axes[0].set_title('Топ-10 брендов, шт.')
axes[1].barh([f'{b} {m}' for b, m in top_md.index][::-1], top_md.values[::-1], color='indianred')
axes[1].set_title('Топ-10 моделей, шт.')
plt.tight_layout()
print(top_br.head(5).round(0).to_string())

In [ ]:
# 3.4 Структура рынка: сегменты, топливо, КПП, привод
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['segment_2013'].value_counts().plot.bar(ax=axes[0], color='slateblue', title='Сегментация 2013')
df['fuel_type'].value_counts().plot.bar(ax=axes[1], color='olivedrab', title='Тип топлива')
df['gearbox'].value_counts().plot.bar(ax=axes[2], color='darkorange', title='Коробка передач')
plt.tight_layout()
print(df['segment_2013'].value_counts(normalize=True).round(3).to_string())
print(df['gearbox'].value_counts(normalize=True).round(3).to_string())

In [ ]:
# 3.5 Динамика рынка по месяцам
monthly = df.groupby(df['sale_date'].dt.to_period('M'))['quantity'].sum()
plt.figure(figsize=(10, 4))
monthly.plot(marker='o', color='steelblue')
plt.title('Продажи рынка по месяцам, шт.')
plt.grid(alpha=0.3)
print(monthly.to_string())

In [ ]:
# 3.6 География продаж
top_reg = df.groupby('region')['quantity'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(9, 4))
top_reg.plot.bar(color='teal')
plt.title('Топ-10 регионов, шт.')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
print((top_reg / df['quantity'].sum() * 100).round(1).to_string())

## 4. Анализ позиции «Меркур Авто»

In [ ]:
# 4.1 Ключевые показатели «Меркур Авто» против рынка
mer = df[df['dealer_company'] == 'Mercur Auto']
units_mkt = df['quantity'].sum()
usd_mkt   = df['total_usd'].sum()

kpi = {
    'Продажи, шт.': (int(mer['quantity'].sum()), int(units_mkt)),
    'Доля рынка, % (шт.)': (round(mer['quantity'].sum() / units_mkt * 100, 2), 100),
    'Выручка, млн USD': (round(mer['total_usd'].sum() / 1e6, 1), round(usd_mkt / 1e6, 1)),
    'Доля рынка, % (USD)': (round(mer['total_usd'].sum() / usd_mkt * 100, 2), 100),
    'Средний чек, USD': (round(mer['total_usd'].sum() / mer['quantity'].sum()),
                         round(usd_mkt / units_mkt)),
    'Средняя цена сделки, USD': (round(mer['price_usd'].mean()), round(df['price_usd'].mean())),
}
pd.DataFrame(kpi, index=['Меркур Авто', 'Рынок']).T

In [ ]:
# 4.2 Структура продаж «Меркур Авто» по брендам и доля бренда на рынке
mer_br = mer.groupby('brand')['quantity'].sum().sort_values(ascending=False)
mkt_br = df.groupby('brand')['quantity'].sum()
brand_share = pd.DataFrame({
    'Меркур, шт.': mer_br,
    'Рынок, шт.': mkt_br.reindex(mer_br.index),
})
brand_share['Доля бренда, %'] = (brand_share['Меркур, шт.'] / brand_share['Рынок, шт.'] * 100).round(1)
brand_share

In [ ]:
# 4.3 Сравнение структуры сегментов
seg_cmp = pd.DataFrame({
    'Меркур, %': mer['segment_2013'].value_counts(normalize=True).round(3) * 100,
    'Рынок, %': df['segment_2013'].value_counts(normalize=True).round(3) * 100})
seg_cmp.plot.bar(figsize=(9, 4), color=['indianred', 'steelblue'])
plt.title('Структура продаж по сегментам, %')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
print(seg_cmp.round(1).to_string())

In [ ]:
# 4.4 Динамика продаж «Меркур Авто» против рынка
mer_monthly = mer.groupby(mer['sale_date'].dt.to_period('M'))['quantity'].sum()
fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(marker='o', label='Рынок (норм.)', color='steelblue', ax=ax)
(mer_monthly / mer_monthly.max() * monthly.max()).plot(marker='s', label='Меркур (норм.)', color='indianred', ax=ax)
plt.title('Динамика продаж (нормировано к максимуму)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
print('Меркур по месяцам, шт.:', mer_monthly.to_dict())

## 5. Выводы

- Рынок КЗ за 9 мес. 2019: **30 336 авто на 677,6 млн USD**; средняя цена сделки 27 893 USD, медиана 20 700.
- Лидеры рынка: **Lada (36,6%), Hyundai (11,4%), Toyota (8,0%)**; топ-модели — Granta, Vesta, Lada 4x4.
- **«Меркур Авто»**: 376 шт. (**1,24%** рынка в штуках) и 14,7 млн USD (**2,17%** в деньгах) — средняя цена сделки 39 273 USD, **на 41% выше рыночной**: премиальное позиционирование работает.
- По брендам: VW — 278 шт. (около 27% рынка бренда), **Audi и Porsche — 100% доли** (монополия в стране).
- Слабые зоны: коммерческий транспорт (0,5% структуры против 13,4% на рынке), концентрация продаж в Алматы (около 63%).
- Ограничение данных: после мая продажи Меркура падают с 66 до 8–14 шт./мес. при стабильном рынке — вероятно, неполнота данных за июнь–сентябрь.